In [13]:
! pip install -U git+https://github.com/StatBiomed/SpatialCV

  Cloning https://github.com/StatBiomed/SpatialCV to /tmp/pip-req-build-p4hxda_j
  Running command git clone -q https://github.com/StatBiomed/SpatialCV /tmp/pip-req-build-p4hxda_j
  Resolved https://github.com/StatBiomed/SpatialCV to commit 67da2e8127b692fe40ba4826c458f2e92e5ccead


In [14]:
%pylab inline
import pandas as pd

rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False

import NaiveDE
import SpatialDE

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [15]:
spot_coord = pd.read_csv("/data2/r10user3/Spatial_Gene_Cell_Ratio/data/human_lung_cell2location/WSA_LngSP8759312/spots.csv", index_col=0)
spot_coord.head()

,X,Y
spot_id,,
WSA_LngSP8759312_AAACAATCTACTAGCA-1,5991,1875
WSA_LngSP8759312_AAACAGAGCGACTCCT-1,11578,3975
WSA_LngSP8759312_AAACAGTGTTCCTGGG-1,5983,15222
WSA_LngSP8759312_AAACCGTTCGTCCAGG-1,5876,11218
WSA_LngSP8759312_AAACCTAAGCAGCCGG-1,10367,13699


In [16]:
spot_coord['X'] = spot_coord['X'] / 112
spot_coord['Y'] = spot_coord['Y'] / 112
spot_coord

,X,Y
spot_id,,
WSA_LngSP8759312_AAACAATCTACTAGCA-1,53.491071,16.741071
WSA_LngSP8759312_AAACAGAGCGACTCCT-1,103.375000,35.491071
WSA_LngSP8759312_AAACAGTGTTCCTGGG-1,53.419643,135.910714
WSA_LngSP8759312_AAACCGTTCGTCCAGG-1,52.464286,100.160714
WSA_LngSP8759312_AAACCTAAGCAGCCGG-1,92.562500,122.312500
...,...,...
WSA_LngSP8759312_TTGTTCAGTGTGCTAC-1,74.017857,52.500000
WSA_LngSP8759312_TTGTTTCACATCCAGG-1,52.455357,110.375000
WSA_LngSP8759312_TTGTTTCATTAGTCTA-1,40.714286,113.767857


In [17]:
cell_abundance = pd.read_csv("/data2/r10user3/Spatial_Gene_Cell_Ratio/data/human_lung_cell2location/WSA_LngSP8759312/cell_ratio.csv", index_col=0)
cell_abundance = cell_abundance.loc[spot_coord.index]
cell_abundance.head()

,q05cell_abundance_w_sf_AT1,q05cell_abundance_w_sf_AT2,q05cell_abundance_w_sf_B_memory,q05cell_abundance_w_sf_B_naive,q05cell_abundance_w_sf_B_plasma_IgA,q05cell_abundance_w_sf_B_plasma_IgG,q05cell_abundance_w_sf_B_plasmablast,q05cell_abundance_w_sf_Basal,q05cell_abundance_w_sf_CD4_EM_Effector,q05cell_abundance_w_sf_CD4_TRM,...,q05cell_abundance_w_sf_SMG_Duct,q05cell_abundance_w_sf_SMG_Mucous,q05cell_abundance_w_sf_SMG_Serous,q05cell_abundance_w_sf_Schwann_Myelinating,q05cell_abundance_w_sf_Schwann_nonmyelinating,q05cell_abundance_w_sf_Secretory_Club,q05cell_abundance_w_sf_Secretory_Goblet,q05cell_abundance_w_sf_Suprabasal,q05cell_abundance_w_sf_T_reg,q05cell_abundance_w_sf_gdT
spot_id,,,,,,,,,,,,,,,,,,,,,
WSA_LngSP8759312_AAACAATCTACTAGCA-1,0.016766,0.014076,0.010948,0.011597,0.022955,0.003584,0.003133,0.008764,0.018236,0.017649,...,0.406095,0.003685,0.019356,0.008968,0.060004,0.005917,0.005824,0.008561,0.011978,0.015547
WSA_LngSP8759312_AAACAGAGCGACTCCT-1,0.001576,0.001234,0.004576,0.005850,0.006346,0.000929,0.002615,0.013344,0.005667,0.004410,...,0.552655,0.018073,0.021225,0.003578,0.004446,0.000840,0.010200,0.011040,0.004703,0.005231
WSA_LngSP8759312_AAACAGTGTTCCTGGG-1,0.018863,0.013069,0.015108,0.013247,0.048117,0.008070,0.009845,0.006468,0.018687,0.012785,...,0.347274,0.023438,0.337589,0.018391,0.051412,0.004230,0.003764,0.002583,0.012940,0.018572
WSA_LngSP8759312_AAACCGTTCGTCCAGG-1,0.121999,0.085840,0.181747,0.234129,2.210338,0.240771,0.348799,0.743238,0.351794,0.351823,...,5.090553,1.506290,4.247349,0.746369,0.852516,0.131831,1.603586,0.696914,0.301343,0.355919
WSA_LngSP8759312_AAACCTAAGCAGCCGG-1,0.032837,0.023461,0.012996,0.016226,0.214170,0.002465,0.006083,0.043069,0.016215,0.015261,...,0.298523,0.023164,0.058677,0.008130,0.015007,0.002763,0.478401,0.042639,0.014131,0.014239


In [18]:
location = np.matrix(spot_coord.values)
location

matrix([[ 53.49107143,  16.74107143],
        [103.375     ,  35.49107143],
        [ 53.41964286, 135.91071429],
        ...,
        [ 40.71428571, 113.76785714],
        [ 37.79464286,  88.23214286],
        [ 61.3125    ,  23.55357143]])

In [19]:
Xdense = np.matrix(cell_abundance.values).T
Xdense

matrix([[0.01676582, 0.00157619, 0.01886272, ..., 0.0392263 , 0.10054886,
         0.0029236 ],
        [0.01407607, 0.00123367, 0.01306905, ..., 0.00870691, 0.15089208,
         0.00134643],
        [0.01094778, 0.0045763 , 0.01510843, ..., 0.10508776, 0.0327964 ,
         0.00875169],
        ...,
        [0.0085613 , 0.01104023, 0.00258274, ..., 0.03679424, 0.07527961,
         0.00370749],
        [0.0119782 , 0.00470316, 0.01294004, ..., 0.15798515, 0.03448979,
         0.00862438],
        [0.01554698, 0.00523094, 0.01857236, ..., 0.17102175, 0.03697508,
         0.01004438]])

In [20]:
import spatialcv

res = spatialcv.sparkx(location, Xdense)
res

,projection,gauss1,gauss2,gauss3,gauss4,gauss5,cosine1,cosine2,cosine3,cosine4,cosine5
0,0.0,0.001640,0.000177,0.000080,0.000083,0.000039,0.284182,0.407064,0.332634,0.063592,0.040429
1,0.0,0.032733,0.092266,0.173639,0.211274,0.220032,0.806912,0.389978,0.161690,0.684413,0.009610
2,0.0,0.000082,0.000000,0.000000,0.000000,0.000000,0.648008,0.008224,0.017393,0.000000,0.000000
3,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.787435,0.000072,0.941020,0.000000,0.000000
4,0.0,0.112564,0.000000,0.000000,0.000000,0.000000,0.302846,0.001926,0.000072,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
75,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.273106,0.000000,0.101282,0.000000,0.000000
76,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.902344,0.000085,0.130389,0.000000,0.000000
77,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.948591,0.000067,0.006062,0.000000,0.000000
78,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.273448,0.000222,0.578815,0.000000,0.000000


In [21]:
# check the real value

from decimal import Decimal

str(Decimal(res['projection'][0]))

'0'

In [22]:
res

,projection,gauss1,gauss2,gauss3,gauss4,gauss5,cosine1,cosine2,cosine3,cosine4,cosine5
0,0.0,0.001640,0.000177,0.000080,0.000083,0.000039,0.284182,0.407064,0.332634,0.063592,0.040429
1,0.0,0.032733,0.092266,0.173639,0.211274,0.220032,0.806912,0.389978,0.161690,0.684413,0.009610
2,0.0,0.000082,0.000000,0.000000,0.000000,0.000000,0.648008,0.008224,0.017393,0.000000,0.000000
3,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.787435,0.000072,0.941020,0.000000,0.000000
4,0.0,0.112564,0.000000,0.000000,0.000000,0.000000,0.302846,0.001926,0.000072,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
75,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.273106,0.000000,0.101282,0.000000,0.000000
76,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.902344,0.000085,0.130389,0.000000,0.000000
77,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.948591,0.000067,0.006062,0.000000,0.000000
78,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.273448,0.000222,0.578815,0.000000,0.000000


In [23]:
df = pd.DataFrame()
df['cell_type'] = cell_abundance.columns
df['p_value'] = res.mean(axis=1)
df.sort_values('p_value')
df.sort_values('p_value').to_csv('/data2/r10user3/Spatial_Gene_Cell_Ratio/code/raw_data/aa.csv')

In [24]:
_# load multi-task GNN cell pearson list
import pickle

with open('/data2/r10user3/Spatial_Gene_Cell_Ratio/code/pearson_log/cell_gene_2layersimpleGNN_lr1e-4_cell_pearson_list.pkl', 'rb') as f:
    multi_cell_pearson_list = pickle.load(f)
df['pearson'] = multi_cell_pearson_list
df

,cell_type,p_value,pearson
0,q05cell_abundance_w_sf_AT1,0.102720,0.164479
1,q05cell_abundance_w_sf_AT2,0.252959,0.140642
2,q05cell_abundance_w_sf_B_memory,0.061246,0.313251
3,q05cell_abundance_w_sf_B_naive,0.157139,0.418429
4,q05cell_abundance_w_sf_B_plasma_IgA,0.037946,0.358119
...,...,...,...
75,q05cell_abundance_w_sf_Secretory_Club,0.034035,0.241010
76,q05cell_abundance_w_sf_Secretory_Goblet,0.093893,0.482241
77,q05cell_abundance_w_sf_Suprabasal,0.086793,0.379403
78,q05cell_abundance_w_sf_T_reg,0.077499,0.468492


In [25]:
df.sort_values('p_value').to_csv('/data2/r10user3/Spatial_Gene_Cell_Ratio/code/raw_data/SpatialCV_meanPvalue.csv')